# Fake News Detection

In [ ]:
!pip install transformers scikit-learn torch gradio

In [ ]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import gradio as gr


## Dataset

In [ ]:
real = [
 'The government announced a new policy.',
 'Scientists discovered a vaccine.',
 'The court issued a ruling.',
]
fake = [
 'Aliens control earth.',
 'Dinosaurs found alive.',
 'People can fly naturally.'
]
df = pd.DataFrame({'text': real + fake, 'label': [0]*len(real) + [1]*len(fake)})

## Tokenization

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
train_texts, test_texts, train_labels, test_labels = train_test_split(df['text'], df['label'], test_size=0.2)
train_enc = tokenizer(list(train_texts), padding=True, truncation=True)
test_enc = tokenizer(list(test_texts), padding=True, truncation=True)

## Dataset Class

In [ ]:
class Dataset(torch.utils.data.Dataset):
 def __init__(self, enc, labels): self.enc = enc; self.labels = labels
 def __getitem__(self, idx):
  item = {k: torch.tensor(v[idx]) for k,v in self.enc.items()}
  item['labels'] = torch.tensor(self.labels.iloc[idx])
  return item
 def __len__(self): return len(self.labels)

train_ds = Dataset(train_enc, train_labels)
test_ds = Dataset(test_enc, test_labels)

## Model + Training

In [ ]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
training_args = TrainingArguments(output_dir='./results', num_train_epochs=4, per_device_train_batch_size=4)
trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=test_ds)
trainer.train()

## Memory + Feedback

In [ ]:
memory = {}
feedback = []

## Prediction + Confidence + Reasoning

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

def explain(text):
 text_lower = text.lower()
 if text_lower in memory:
  return 'Prediction based on user feedback memory (instant learning override).'
 return 'Prediction based on learned language patterns (tone, structure, and context).'

def predict(text):
 text_lower = text.lower()

 if text_lower in memory:
  label = memory[text_lower]
  return f"{'REAL' if label==0 else 'FAKE'} (memory override)"

 inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True)
 inputs = {k:v.to(device) for k,v in inputs.items()}

 with torch.no_grad():
  outputs = model(**inputs)

 probs = torch.softmax(outputs.logits, dim=1)
 pred = torch.argmax(probs, dim=1).item()
 confidence = probs[0][pred].item()

 return f"{'REAL' if pred==0 else 'FAKE'} (confidence: {confidence:.2f})"

## Feedback Functions

In [ ]:
def add_feedback(text, label):
 feedback.append({'text': text, 'label': label})
 memory[text.lower()] = label
 return 'Feedback stored + instant learning applied'

def retrain():
 global df
 if not feedback:
  return 'No feedback yet'
 df = pd.concat([df, pd.DataFrame(feedback)], ignore_index=True)
 trainer.train()
 feedback.clear()
 return 'Model retrained with feedback'

## Gradio UI (Glass Style + Reasoning Button)

In [ ]:
css = '''
body {background: linear-gradient(135deg, #0f172a, #1e1b4b);}
.gradio-container {background: rgba(255,255,255,0.05); backdrop-filter: blur(20px); border-radius: 20px; padding: 20px;}
button {border-radius: 12px !important;}
'''

with gr.Blocks(css=css) as demo:
 gr.Markdown('# Adaptive AI — Fake News Detector')
 input_box = gr.Textbox(label='Enter News')
 output_box = gr.Textbox(label='Prediction')
 reason_box = gr.Textbox(label='Why this prediction?')

 gr.Button('Predict').click(predict, inputs=input_box, outputs=output_box)
 gr.Button('Why this prediction?').click(explain, inputs=input_box, outputs=reason_box)

 gr.Markdown('### Feedback')
 label = gr.Radio(['REAL','FAKE'], label='Correct Label')
 gr.Button('Submit Feedback').click(lambda t,l: add_feedback(t,0 if l=='REAL' else 1), inputs=[input_box,label], outputs=output_box)

 gr.Button('Retrain Model').click(retrain, outputs=output_box)

demo.launch()